In [ ]:
library(Seurat)
library(dplyr)
library(data.table)
library(ggplot2)
source("~/Projects/heads/clustering.r")

In [ ]:
data_dir = "/gpfs/gibbs/pi/braun/zy325"

In [ ]:
### Miya ###

obj_path = "/gpfs/gibbs/pi/braun/mh2632/SC_RCC_Individual_RDS/SC_RCC_merged_20230612.rds"

# obj_path = "/vast/palmer/scratch/kleinstein/zy325/SC_RCC_int_merged_v5_anno.rds"

obj = readRDS(obj_path)

In [ ]:
meta = obj@meta.data
meta = meta[,-grep("pANN_|DF.classifications_",colnames(meta))]

meta = meta %>% mutate(
    name=rownames(meta),
    sample_id3=gsub("_PostQC.*$","",name),
    sample_id3=gsub("_new_mt","",sample_id3))

meta$sample_id3[meta$sample_id3 == "SC_07_NORM"] = "SC_07_NK"
meta$orig.ident[meta$sample_id3 == "SC_64_NORM"] = "SC_64_NORM"

meta = meta %>% mutate(
    sample_id1 = gsub("SC_","SC_RCC_",sample_id3),
    sample_id2 = gsub("_","",sample_id1)) %>%
    select(
        orig.ident,nCount_RNA,nFeature_RNA,percent.mt,name,
        sample_id1,sample_id2,sample_id3,DoubletScore,DoubletCall)

obj@meta.data = meta

In [ ]:
### Batch ###

batch = as.data.frame(fread(file.path(data_dir,"metadata","batch.csv")))

meta = left_join(obj@meta.data,batch,by="sample_id1")
rownames(meta) = rownames(obj@meta.data)

obj@meta.data = meta

In [ ]:
### Sample-level QC ###
# Based on web summary (cellranger)
# Samples are filtered due to
# 1. Median genes per cell < 500
# 2. Estimated number of cells < 100
# 3. Not kidney tumor

rm_samples = paste0("SC_RCC_",c("08","17","25","34","63","77",
                                "69"))

obj = obj[,-which(obj$sample_id1 %in% rm_samples)]

In [ ]:
obj[["RNA"]] = as(obj[["RNA"]],"Assay5")

obj = clustering(obj,
                plot_QC_metrics = F,
                group.by.vars = "batch_lab",
                harmony_theta = 1,dims = 1:50)

In [ ]:
#saveRDS(obj,file=file.path(data_dir,"processed","theta1_dims50","scrcc_miya_clustered_.rds"))

obj = readRDS(file.path(data_dir,"processed","theta1_dims50","scrcc_miya_clustered.rds"))

In [ ]:
options(repr.plot.width=8,repr.plot.height=8)
DimPlot(obj,label = T) + NoLegend()

In [ ]:
options(repr.plot.width=16,repr.plot.height=8)
DimPlot(obj,group.by = "batch_lab") | DimPlot(obj,group.by = "batch_seq_rna") #+ NoLegend()

In [ ]:
options(repr.plot.width=15,repr.plot.height=10)
VlnPlot(obj,features = c("MZB1","JCHAIN","SDC1","CSF3R","FPR1","FCGR3B","TPSAB1","CPA3","MS4A2","HBB","PPBP","EPCAM","ALDOB","PLVAP","ACTA2","PTPRC"),pt.size = 0,stack = T,flip = T)

In [ ]:
############ theta1_dims50 ############

# Find CD45+ clusters
obj$CD45_data = obj@assays$RNA@layers$data[which(rownames(obj)=="PTPRC"),]

immune_clusters = obj@meta.data %>% group_by(seurat_clusters) %>% 
    summarise(CD45_data_q3 = quantile(CD45_data,probs = .75)) %>% 
    filter(CD45_data_q3>0) %>% .$seurat_clusters

# Check if any CD45low immune cells are missed out
# PC - MZB1, JCHAIN, SDC1
# Mast - TPSAB1, CPA3, MS4A2
# Neutrophil - CSF3R, FPR1, FCGR3B

# Check if any CD45+ clusters are contamination
# High RBC - 27
# High Epi - 37,47,56
# High Endo - 44
dbl_df = data.frame(
    cluster = c(27,37,44,47,56),
    annotation = c("Immune-RBC",rep("Immune-Epi",3),"Immune-Endo")
)

immune_clusters = setdiff(immune_clusters,dbl_df$cluster)

In [ ]:
############ theta3_dims30 ############

# Find CD45+ clusters
obj$CD45_data = obj@assays$RNA@layers$data[which(rownames(obj)=="PTPRC"),]

immune_clusters = obj@meta.data %>% group_by(seurat_clusters) %>% 
    summarise(CD45_data_q3 = quantile(CD45_data,probs = .75)) %>% 
    filter(CD45_data_q3>0) %>% .$seurat_clusters

# Check if any CD45low immune cells are missed out
# PC - MZB1, JCHAIN, SDC1
# Mast - TPSAB1, CPA3, MS4A2
# Neutrophil - CSF3R, FPR1, FCGR3B

# Check if any CD45+ clusters are contamination
# High RBC - 15
# High Epi - 20
# Small clusters - 49,53,54,55

immune_clusters = setdiff(immune_clusters,c(15,20,49,53,54,55))

In [ ]:
obj$lineage1 = case_when(
    obj$`RNA_snn_res.0.5` %in% immune_clusters ~ "Immune",
    obj$`RNA_snn_res.0.5` %in% dbl_df$cluster ~ "contamination",
    TRUE~"NonImmune") 

In [ ]:
### Doublet collection ###

dbls = list()
dbl = subset(obj, `RNA_snn_res.0.5` %in% dbl_df$cluster)
dbl$anno_dbl = dbl_df$annotation[match(dbl$`RNA_snn_res.0.5`,dbl_df$cluster)]
dbls[["miya"]] = dbl
saveRDS(dbls,file = file.path(data_dir,"processed","theta1_dims50","dbl_list.rds"))

In [ ]:
options(repr.plot.width=16,repr.plot.height=8)
DimPlot(obj,group.by = "lineage1") + NoLegend() | FeaturePlot(obj,features= "PTPRC") #+ NoLegend()

In [ ]:
cd8t = readRDS("/gpfs/gibbs/project/braun/zy325/scrcc/processed/seurat_objects/Clustering_CD8T_0912.rds")
cd8t = subset(cd8t, sample!="SCRCC69")
#cd8t$tissue = ifelse(grepl("NORM|NK",cd8t$sample),"normal","tumor")
#cd8t = subset(cd8t, tissue=="tumor")
table(cd8t$Round2_cls)

obj$anno_cd8t = unname(cd8t$Round2_cls[match(obj$name,cd8t$name)])
# obj$anno_cd8t[is.na(obj$anno_cd8t)] = "filtered"

library(ggsankey)
obj@meta.data %>% 
    filter(!is.na(anno_cd8t)) %>%
    make_long(lineage1,anno_cd8t) %>% # `RNA_snn_res.0.5`
    ggplot(aes(x = x, 
               next_x = next_x, 
               node = node, 
               next_node = next_node,
               fill = factor(node),
               label = node)) +
    geom_sankey(flow.alpha = 0.5, node.color = 1) +
    geom_sankey_label(size = 3, color = 1, fill = "white") +
    theme_sankey(base_size = 16) + NoLegend()

table(obj$lineage1[obj$anno_cd8t == "CD8Tex_NMF3"]) %>% sort(decreasing = T)

In [ ]:
### Previous annotation ###

# main = readRDS("/gpfs/gibbs/project/braun/zy325/scrcc/processed/seurat_objects/Clustering_main_0926.rds")
main = readRDS("/gpfs/gibbs/pi/braun/zy325/scrcc/processed/seuratobj_2023/Clustering_main_0926.rds")

cts = unique(main$Round2_cls)

cd8t = cts[grepl("^(CD8T|MAIT)",cts)]
cd4t = cts[grepl("^CD4T",cts)]
nk = cts[grepl("^NK",cts)]
ilc = cts[grepl("^(gdT|ILC)",cts)]
b = cts[grepl("^(B|PC|Doublet)",cts)]
lym = c(cd8t,cd4t,nk,ilc,b)

mye = cts[grepl("^(Mono|Macro|cDC|pDC|Mast|Mye)",cts)]

fibro = cts[grepl("CAF|Fibro|Pericyte|SMC",cts)]
endo = cts[grepl("Art|Vein|Tip|Lym|Cap",cts)]
epi = cts[grepl("Epi",cts)]

imm = c(lym,mye)
nonimm = c(endo,epi,fibro)

obj$anno1 = case_when(
    obj$name %in% main$name[main$Round2_cls %in% imm] ~ "immune",
    obj$name %in% main$name[main$Round2_cls %in% nonimm] ~ "nonimmune",
    TRUE~"filtered"
)

obj$anno2 = case_when(
    obj$name %in% main$name[main$Round2_cls %in% lym] ~ "lymphoid",
    obj$name %in% main$name[main$Round2_cls %in% mye] ~ "myeloid",
    obj$name %in% main$name[main$Round2_cls %in% endo] ~ "endothelial",
    obj$name %in% main$name[main$Round2_cls %in% fibro] ~ "fibroblast",
    obj$name %in% main$name[main$Round2_cls %in% epi] ~ "epithelial",
    TRUE~"filtered"
)

obj$anno3 = case_when(
    obj$name %in% main$name[main$Round2_cls %in% cd4t] ~ "cd4t",
    obj$name %in% main$name[main$Round2_cls %in% cd8t] ~ "cd8t",
    obj$name %in% main$name[main$Round2_cls %in% nk] ~ "nk",
    obj$name %in% main$name[main$Round2_cls %in% ilc] ~ "ilc",
    obj$name %in% main$name[main$Round2_cls %in% b] ~ "b",
    obj$name %in% main$name[main$Round2_cls %in% mye] ~ "myeloid",
    obj$name %in% main$name[main$Round2_cls %in% endo] ~ "endothelial",
    obj$name %in% main$name[main$Round2_cls %in% fibro] ~ "fibroblast",
    obj$name %in% main$name[main$Round2_cls %in% epi] ~ "epithelial",
    TRUE~"filtered"
)

obj$anno4 = unname(main$Round2_cls[match(obj$name,main$name)])
obj$anno4[is.na(obj$anno4)] = "filtered"


In [ ]:
obj@meta.data %>% 
    make_long(lineage1,anno1) %>% # `RNA_snn_res.0.5`
    ggplot(aes(x = x, 
               next_x = next_x, 
               node = node, 
               next_node = next_node,
               fill = factor(node),
               label = node)) +
    geom_sankey(flow.alpha = 0.5, node.color = 1) +
    geom_sankey_label(size = 3, color = 1, fill = "white") +
    theme_sankey(base_size = 16) + NoLegend()


In [ ]:
obj@meta.data %>% head

In [ ]:
obj$lineage1_clusters = obj$`RNA_snn_res.0.5`
obj$`RNA_snn_res.0.5` = NULL
obj$seurat_clusters = NULL

In [ ]:
obj = subset(obj,lineage1=="Immune")

In [ ]:
saveRDS(obj,file=file.path(data_dir,"processed","theta1_dims50","scrcc_immune.rds"))

In [ ]:
m = FindMarkers(obj,`ident.1` = 39,only.pos = T,logfc.threshold = 1)
m %>% filter(p_val_adj<0.01) %>% arrange(desc(avg_log2FC)) %>% filter(abs(pct.1-pct.2)>.1)

In [ ]:
### Trace Tex-rm ###
obj$anno_int = obj$lineage1

obj1 = readRDS("/gpfs/gibbs/pi/braun/zy325/scrcc/processed/seuratobj_2026/scrcc_myeloid.rds")
obj$anno_int[obj$name %in% colnames(obj1)] = "myeloid"

obj2 = readRDS("/gpfs/gibbs/pi/braun/zy325/scrcc/processed/seuratobj_2026/scrcc_lymphoid_clustered_addCgenes.rds")
obj$anno_int[obj$name %in% colnames(obj2)] = "lymphoid"
obj$anno_int[obj$anno_int == "Immune"] = "contamination"

In [ ]:
Tcell = c(0:3,5,8,11,12,15)
B = c(9,13,16)
NK = c(6,10,14)
ILC = c(4)

# Contamination
# Hypoxic: SPP1+ VEGFA+ - 7
# Small clusters - 17

obj2$lineage3 = case_when(
    obj2$`RNA_snn_res.0.5` %in% Tcell~"T",
    obj2$`RNA_snn_res.0.5` %in% B~"B",
    obj2$`RNA_snn_res.0.5` %in% NK~"NK",
    obj2$`RNA_snn_res.0.5` %in% ILC~"ILC",
    TRUE~"contamination")

obj$anno_int[match(obj2$name,obj$name)] = obj2$lineage3

In [ ]:
#obj3 = readRDS("/gpfs/gibbs/pi/braun/zy325/scrcc/processed/seuratobj_2026/scrcc_t_ilc2t_clustered.rds")
#obj3 = FindClusters(obj3,resolution = 0.3)

cd8t = c(1,2,5,6,8)
cd4t = c(0,3,7)
cyclingt = c(4)

obj3$lineage3.5 = case_when(
    obj3$`RNA_snn_res.0.3` %in% cd8t ~ "CD8T",
    obj3$`RNA_snn_res.0.3` %in% cd4t ~ "CD4T",
    obj3$`RNA_snn_res.0.3` %in% cyclingt  ~ "CyclingT",
    TRUE~"contamination")

obj$anno_int[match(obj3$name,obj$name)] = obj3$lineage3.5

In [ ]:
obj4 = readRDS("/gpfs/gibbs/pi/braun/zy325/scrcc/processed/seuratobj_2026/scrcc_cd8t_annotated.rds")

obj4$lineage5 = obj4$lineage4
tex_id = which(obj4$lineage5 == "CD8Tex_PDCD1")
obj4$lineage5[tex_id] = obj4$anno_tex[tex_id]
obj4$lineage5[is.na(obj4$lineage5)] = "Tex_ANT"
obj$anno_int[match(obj4$name,obj$name)] = obj4$lineage5

In [ ]:
library(ggsankey)
options(repr.plot.width=7, repr.plot.height=10)

df <- obj@meta.data %>%
    filter(anno_cd8t == "CD8Tex_NMF3") %>%
    make_long(anno_int, anno_cd8t)

df <- df %>%
    group_by(x, node) %>%
    mutate(n = n()) %>%
    ungroup() %>%
    mutate(label_n = paste0(node, " (", n, ")"))

node_order <- df %>%
    distinct(node, n) %>%
    group_by(node) %>%
    summarise(n = max(n), .groups = "drop") %>%
    arrange(n) %>%
    pull(node)

df <- df %>%
    mutate(node = factor(node, levels = node_order),
           next_node = factor(next_node, levels = node_order))

df %>%
    ggplot(aes(x = x,
               next_x = next_x,
               node = node,
               next_node = next_node,
               fill = node,
               label = label_n)) +
    geom_sankey(flow.alpha = 0.5, node.color = 1) +
    geom_sankey_label(size = 3.5, color = 1, fill = "white", hjust = 1) +
    theme_sankey(base_size = 16) + NoLegend()

In [ ]:
write.table(obj@meta.data %>% select(name,anno_int),file = "../lymphoid_addCgenes/anno_int.txt",quote = F,sep = "\t",row.names = F)